# COMP3122 — Week 3 Demo Notebook
### Chapter 3: NumPy — Aggregation, Broadcasting, and Selection

**Applied Data Science**

This notebook is the live-coding companion to the Week 3 lecture. It follows the same six
blocks as the slides, in the same order, on the same temperature dataset.

Run it top to bottom. Every cell depends on the ones above it — in particular on `temps`,
which is built once in Setup and used for the rest of the notebook.

Three cells are marked **LIVE DEMO**. Those are the ones the lecture stops for.

---
## Setup

One import, and the dataset. Four cities down the side, seven days across.

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)

In [ ]:
# explain: daily high temperatures in degrees Celsius, 4 cities across 7 days
temps = np.array([
    [ 8, 10, 12, 11,  9, 13, 14],   # Toronto [0]
    [22, 23, 21, 24, 25, 26, 24],   # Cairo   [1]
    [15, 16, 14, 17, 18, 16, 17],   # Tokyo   [2]
    [20, 22, 21, 23, 22, 21, 20],   # Sydney  [3]
])
# columns: Mon [0]  Tue [1]  Wed [2]  Thu [3]  Fri [4]  Sat [5]  Sun [6]

print(temps)
print(temps.shape)          # explain: (4, 7) — four rows, seven columns

---
## Block 1 — Asking the whole array a question  ·  Book §3.1

An **aggregation** replaces many numbers with one number. Every one of these collapses all
28 readings down to a single value.

In [ ]:
print(temps.sum())        # explain: add up all 28 readings → 504
print(temps.mean())       # explain: the average of all 28 → 18.0
print(temps.min())        # explain: the coldest single reading → 8
print(temps.max())        # explain: the warmest single reading → 26
print(temps.std())        # explain: how spread out the values are → 5.085
print(np.median(temps))   # explain: the middle value when all 28 are sorted → 19.0

**The parentheses are not optional.** `sum`, `mean`, `min`, `max` and `std` are *methods* —
they do work each time you call them. `shape` from last week is an *attribute* and has no
parentheses. Leave the parentheses off a method and you get this:

In [ ]:
print(temps.sum)          # explain: no parentheses — this is NOT a number

# You will see something like:
#   <built-in method sum of numpy.ndarray object at 0x...>
# That is not an error message. It is Python showing you the method itself,
# because you asked for the method rather than calling it.

`median` is the odd one out: it exists only as `np.median(arr)`, never as `arr.median()`.

In [ ]:
print(np.median(temps))          # explain: this works
# print(temps.median())          # explain: this does NOT — uncomment to see the AttributeError

### Mean or median?

They are not interchangeable. The mean is pulled around by extreme values; the median is not.

In [ ]:
arr = np.array([20, 22, 25, 30, 100])   # explain: four ordinary values and one outlier

print(arr.mean())        # explain: dragged upward by the 100 → 39.4
print(np.median(arr))    # explain: still sitting with the other four → 25.0
print(arr.std())         # explain: large, because 100 is far from the rest → about 30.49

If these were house prices, you would report the median. One mansion should not move the
number that describes everyone else.

### Percentiles

The p-th percentile is the value that roughly p percent of the data sits at or below.

In [ ]:
print(np.percentile(temps, 50))   # explain: the median, by another name → 19.0
print(np.percentile(temps, 90))   # explain: → 24.0
print(np.percentile(temps, 25))   # explain: → 14.0

# Check the shares yourself rather than taking them on trust:
print((temps > 24).sum(), 'readings above 24, out of', temps.size)
print((temps <= 14).sum(), 'readings at or below 14, out of', temps.size)

Two of the 28 are above 24, and eight are at or below 14 — not exactly 10% and 25%.
With 28 values and repeats, the share almost never lands on the round number. NumPy smooths
between neighbouring values, which is also why percentiles are often not whole numbers.

### Aggregating part of the data

Slice first, then aggregate whatever came back. Nothing new — last week's slicing feeding
this week's aggregation.

In [ ]:
print(temps[0].mean())       # explain: Toronto's [0] weekly average → 11.0
print(temps[1].max())        # explain: Cairo's [1] hottest day → 26
print(temps[:, 0].mean())    # explain: Monday's [0] four-city average → 16.25

So how would you get the weekly average for **all four** cities at once? That is Block 2.

---
## Block 2 — Which direction do you collapse?  ·  Book §3.2

> **The axis you pass in is the axis that goes away.**

### LIVE DEMO — watch the shapes

This is the cell the lecture stops for. Do not just read the numbers; read the shapes.

In [ ]:
print('temps.shape        ', temps.shape)          # explain: (4, 7) to start with

print('mean(axis=0)      ', temps.mean(axis=0))
print('  its shape       ', temps.mean(axis=0).shape)   # explain: the 4 went away → (7,)

print('mean(axis=1)      ', temps.mean(axis=1))
print('  its shape       ', temps.mean(axis=1).shape)   # explain: the 7 went away → (4,)

`axis=0` collapsed the four city rows, leaving one value per day.
`axis=1` collapsed the seven day columns, leaving one value per city.

Check the first one by hand: Monday's readings are 8, 22, 15 and 20.
`(8 + 22 + 15 + 20) / 4 = 65 / 4 = 16.25`.

In [ ]:
print((8 + 22 + 15 + 20) / 4)                # explain: 16.25, by hand
print(temps[:, 0].mean())                    # explain: 16.25, by NumPy
print(temps.mean(axis=0)[0])                 # explain: 16.25, the first value of the row

### Every aggregation takes `axis`

In [ ]:
print(temps.max(axis=1))          # explain: each city's hottest day
print(temps.sum(axis=0))          # explain: total degrees per day, across the four cities
print(np.median(temps, axis=1))   # explain: each city's median for the week

### `argmax` and `argmin` — where, not what

In [ ]:
print(temps.max(axis=1))          # explain: the VALUES — 14, 26, 18, 23
print(temps.argmax(axis=1))       # explain: the POSITIONS — 6, 5, 4, 3

days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
cities = ['Toronto', 'Cairo', 'Tokyo', 'Sydney']
for i, pos in enumerate(temps.argmax(axis=1)):
    print(f'{cities[i]:<9} hottest on {days[pos]}')   # explain: mapping positions to names by hand

`argmax` gives you `6`, not `'Sun'`. Mapping a position back to a label is a step you do
yourself for now. Pandas will do it for you in Chapter 4.

---
## Block 3 — How NumPy stretches arrays  ·  Book §3.3

The hardest idea in the course. Take it slowly.

**The rule.** Line the two shapes up *from the right*. For each pair, ask: are they equal,
or is one of them 1? Yes to either and that dimension is fine. No to both and it fails.
If one array has fewer dimensions, pretend it is padded with 1s on the left.

In [ ]:
a = np.array([1, 2, 3])       # explain: shape (3,)
b = np.array([10, 20, 30])    # explain: also (3,) — same shape, no stretching needed
print(a + b)                  # explain: pair up position by position → [11 22 33]

print(a + 100)                # explain: the scalar is stretched to every element
# You have been broadcasting since last week without being told the name.

### Example 1 — subtract the overall mean

In [ ]:
print(temps - 18)   # explain: one scalar, subtracted from all 28 values; result stays (4, 7)

Toronto's row is all negative — colder than average all week. Cairo's is all positive.

### Example 2 — subtract each DAY's average

In [ ]:
daily_mean = temps.mean(axis=0)     # explain: one average per day, across the four cities
print(daily_mean)
print(daily_mean.shape)             # explain: (7,)

#   temps       (4, 7)
#   daily_mean     (7,)  ->  padded to (1, 7)
#   rightmost pair 7 and 7: equal, pass
#   next pair      4 and 1: one is 1, stretch
print(np.round(temps - daily_mean, 2))

The first entry is `-8.25`: Toronto's Monday reading of 8, minus Monday's four-city mean of
16.25. NumPy behaved as if `daily_mean` had been copied down to all four rows. Nothing was
really copied — that is what makes it fast.

### Example 3 — subtract each CITY's average, and the trap

In [ ]:
city_mean = temps.mean(axis=1)      # explain: one average per city, across the seven days
print(city_mean)
print(city_mean.shape)              # explain: (4,)

Four cities, four values. It looks like it should just work. Line the shapes up and it does not:

```
temps        (4, 7)
city_mean       (4,)  ->  padded to (1, 4)
rightmost pair: 7 and 4  ->  different, and neither is 1  ->  FAIL
```

In [ ]:
try:
    temps - city_mean
except ValueError as e:
    print('ValueError:', e)      # explain: run it and read the real message

**The fix** is to make `city_mean` a *column* of four values — shape `(4, 1)` — so the 4
lines up against the row axis:

```
temps               (4, 7)
city_mean.reshape   (4, 1)
rightmost pair: 7 and 1  ->  one is 1, stretch
next pair:      4 and 4  ->  equal
```

In [ ]:
anomalies = temps - city_mean.reshape(4, 1)   # explain: reshape to a COLUMN first
print(np.round(anomalies, 2))

print(np.round(anomalies.mean(axis=1), 10))   # explain: every row averages to zero

Each row now shows how that city's week swung around its own typical week. Every row
averages to zero — that is what subtracting a mean does.

**The shortcut.** `.reshape(-1, 1)` means "work the first size out yourself". Use it so you
do not have to edit the number when your data changes size.

In [ ]:
print(city_mean.reshape(-1, 1).shape)     # explain: (4, 1), without hard-coding the 4

big = np.arange(100)
print(big.reshape(-1, 1).shape)           # explain: (100, 1) — same code, different data

### Example 4 — a real use

In [ ]:
temps_f = temps * 9/5 + 32     # explain: two scalar broadcasts in one line
print(np.round(temps_f, 1))
print(8 * 1.8 + 32)            # explain: Toronto's Monday, 8 C, is 46.4 F

---
## Block 4 — Selecting by condition  ·  Book §3.4

A **boolean mask** is a same-shape array of `True` / `False`, built by a comparison.

In [ ]:
hot = temps > 20
print(hot)
print(hot.shape)          # explain: same shape as temps — (4, 7)
print(hot.dtype)          # explain: bool, one byte per entry

Toronto never crosses 20. Cairo always does. Sydney does except Monday and Sunday — both
are exactly 20, and 20 is not strictly greater than 20.

### LIVE DEMO — why the parentheses matter

This is the second cell the lecture stops for. Run the wrong one first, on purpose.

In [ ]:
# WRONG — no parentheses around each comparison.
try:
    warm = temps >= 21 & temps <= 24
except ValueError as e:
    print('ValueError:', e)

# `&` binds TIGHTER than `>=`, so Python computes  21 & temps  first —
# the number 21 combined with the whole array — before any comparison happens.
print()
print('21 & temps gives:')
print(21 & temps)          # explain: a bitwise AND on every element. Nonsense here, but legal.

In [ ]:
# RIGHT — every comparison in its own brackets.
warm = (temps >= 21) & (temps <= 24)
print(warm.sum(), 'readings are between 21 and 24 inclusive')

# Use & for AND and | for OR. Never the words `and` / `or` — they try to give one
# true-or-false answer for the whole array. `&&` and `||` do not work at all.

### Count with a mask, then select with it

In [ ]:
print((temps > 20).sum())          # explain: True counts as 1 → 12 readings above 20
print((temps > 20).sum(axis=1))    # explain: per city → [0 7 0 5]

print(temps[temps > 20])           # explain: the matching VALUES, always 1-D
print(temps[temps > 20].shape)     # explain: (12,) — the (4, 7) shape is gone
print(temps[temps > 20].mean())    # explain: select first, then aggregate → 22.833...

In [ ]:
print((temps > 25).any())          # explain: is at least one reading above 25? → True
print((temps > 5).all())           # explain: is every reading above 5? → True
print((temps > 20).all())          # explain: is every reading above 20? → False

# sum() answers HOW MANY match; any() and all() answer WHETHER they match.


The result is flat because the twelve matches are scattered — they do not form a rectangle,
so no rectangle can come back.

### `~` flips a mask

In [ ]:
print(temps[~(temps > 20)])    # explain: everything NOT above 20
print(temps[temps <= 20])      # explain: the same result, written the other way

### A mask on the left of `=` changes the data

In [ ]:
temps_capped = temps.copy()               # explain: work on a COPY, not the original
temps_capped[temps_capped > 25] = 25      # explain: clip everything above 25

print(temps_capped[1])                    # explain: Cairo's 26 became 25
print(temps[1])                           # explain: the original is untouched

# Had we written temps[temps > 25] = 25, the original dataset would be gone.
# np.clip does this in one call — the mask version shows the mechanism.

### Fancy indexing — a list of positions

In [ ]:
print(temps[[0, 2]])          # explain: rows 0 and 2 — Toronto and Tokyo. Stays 2-D.
print(temps[[0, 2]].shape)    # explain: (2, 7)

print(temps[:, [0, 6]])       # explain: every row, columns 0 and 6 — Mon and Sun
print(temps[:, [0, 6]].shape) # explain: (4, 2)

print(temps[[3, 1, 0, 2]])    # explain: any order you like — Sydney, Cairo, Toronto, Tokyo

A **mask** when your selection comes from a condition. **Fancy indexing** when it comes from
a list of positions you already know. These examples stay 2-D because whole rows or whole
columns were picked.

---
## Block 5 — Putting values in order  ·  Book §3.5

In [ ]:
toronto = temps[0]            # explain: Toronto's [0] week as a 1-D array
print(toronto)

print(np.sort(toronto))       # explain: the same values, ascending, as a NEW array
print(toronto)                # explain: the original is unchanged

**`axis` behaves differently here.** In an aggregation the axis you pass disappears. In a
sort it does not — the values are reordered along it and the shape comes back unchanged.

In [ ]:
print(np.sort(temps, axis=1))         # explain: each row sorted independently
print(np.sort(temps, axis=1).shape)   # explain: still (4, 7) — nothing collapsed
print(temps.mean(axis=1).shape)       # explain: (4,) — compare: aggregation DOES collapse

### `np.argsort` — where the values were

In [ ]:
print(toronto)                 # explain: [ 8 10 12 11  9 13 14]
print(np.argsort(toronto))     # explain: [0 4 1 3 2 5 6]

# Read it as instructions: the smallest is at position 0, the next at position 4,
# then position 1, and so on.
print(toronto[np.argsort(toronto)])   # explain: fancy indexing with argsort rebuilds the sort

The positions matter when the array is paired with something else. Sort the values and you
throw the names away; sort the positions and you keep them.

In [ ]:
students = np.array(['Amara', 'Ben', 'Chen', 'Dev', 'Eli'])
scores   = np.array([72, 91, 65, 88, 79])

order = np.argsort(scores)[::-1]      # explain: positions, highest score first
for rank, pos in enumerate(order, start=1):
    print(rank, students[pos], scores[pos])   # explain: the names came along for the ride

### Sorting in reverse

In [ ]:
print(np.sort(toronto)[::-1])          # explain: sort ascending, then walk backwards
print(np.array([10, 20, 30])[::-1])    # explain: [::-1] on its own just reverses

# Careful on a 2-D array: [::-1] reverses the ROW ORDER, not the values in each row.
print(np.sort(temps, axis=1)[::-1])    # explain: rows reversed — probably not what you wanted

---
## Block 6 — The same answer twice  ·  Book §3.6

### LIVE DEMO — with a seed, then without

The third cell the lecture stops for. Run the seeded cell twice, then the unseeded one twice.

In [ ]:
np.random.seed(42)             # explain: fix the generator's starting point
print(np.random.rand(5))

np.random.seed(42)             # explain: reset to the SAME starting point
print(np.random.rand(5))

# Identical, every time you run this cell.

In [ ]:
# The NEXT draw from the SAME seeded generator. Run this cell two or three times.
print(np.random.rand(5))

# Each run gives a new set of numbers — but not random ones: the generator is still
# walking the sequence that seed 42 started. Restart the kernel, run the cells above,
# run this once, and you get exactly these five values again (0.1559..., 0.0580..., ...).
# Reproducible means: same seed + same calls in the same order = same numbers.
# Without ANY seed, a fresh kernel would give different numbers every time.


Any integer from 0 up to `2**32 - 1` works as a seed. Go above it and NumPy says so:

In [ ]:
try:
    np.random.seed(2**32)
except ValueError as e:
    print('ValueError:', e)

np.random.seed(2**32 - 1)      # explain: the largest legal seed
print('that one was fine')

### `rand` and `randn` are different shapes of random

In [ ]:
np.random.seed(0)
uniform = np.random.rand(1000)          # explain: every value in [0, 1) equally likely
normal  = np.random.randn(1000)         # explain: clustered around 0, bell-shaped

# 4 decimals, not 3: rand's max here is 0.9998, and rounding it to 3 would print 1.0 —
# which would contradict the half-open [0, 1) range on the line above.
print('rand  min/max/mean:', round(uniform.min(), 4), round(uniform.max(), 4), round(uniform.mean(), 4))
print('randn min/max/mean:', round(normal.min(), 4), round(normal.max(), 4), round(normal.mean(), 4))
print('randn values beyond +/-2:', (np.abs(normal) > 2).sum(), 'out of 1000')

`rand` spreads evenly between 0 and 1. `randn` clusters around 0, can go negative, and only
rarely passes ±2. Use `rand` for "any value in the range", `randn` for "small wiggles".

### The calls you will use, and noise the shape of the data

In [ ]:
np.random.seed(42)                      # explain: seed once; everything below is reproducible

print(np.random.randint(0, 100, 5))     # explain: 5 integers in [0, 100) — 100 EXCLUDED
print(np.random.choice([10, 20, 30], 4))# explain: 4 picks from the list, with replacement

np.random.seed(0)
noise = np.random.randn(4, 7)           # explain: same shape as temps
noisy_temps = temps + noise             # explain: shapes match, so it adds position by position
print(np.round(noisy_temps, 1))

---
## The mindset shift  ·  Book §3.7

> **Most of NumPy's power is in shaping arrays so that the operation you want is the natural one.**

Three loops, and their one-line versions. Run both halves and compare.

In [ ]:
# Before — two nested loops and two accumulators
city_means = []
for row in temps:
    total = 0
    for value in row:
        total += value
    city_means.append(total / len(row))
print(city_means)

# After
print(temps.mean(axis=1))

In [ ]:
# Before — a hand-rolled counter
warm_count = 0
for row in temps:
    for value in row:
        if value > 20:
            warm_count += 1
print(warm_count)

# After
print((temps > 20).sum())

In [ ]:
# Before — twelve lines to build a list of lists
anomalies_loop = []
for row in temps:
    m = sum(row) / len(row)
    anomalies_loop.append([value - m for value in row])

# After
anomalies_np = temps - temps.mean(axis=1).reshape(-1, 1)

print(np.allclose(np.array(anomalies_loop), anomalies_np))   # explain: True — same answer

**The rule.** If you are reaching for a loop over an array, ask three questions.
Is there an aggregation that already does this? Is there a comparison that builds the mask I
need? Is there a reshape that makes the two arrays line up? Usually the answer is yes.

---
## Before next class

Lecture quiz (deadline on the quiz itself) · finish Lab 03 · read Chapter 4.

Chapter 4 introduces **Pandas**, which puts *names* on these rows and columns so you can write
`df.loc['Toronto'].mean()` instead of `temps[0].mean()`. Everything from today carries over — the
same four cities are the first thing you will see.

**If broadcasting is still murky:** re-read §3.3 and redo Quick Check 3.3 on paper, without a
computer. Twenty minutes, and it settles. That is normal, not a failure.